# Data Loading

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\yong-chen.how\Desktop\Chen\Nus\Project\fashion-recommender\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the article metadata
current_folder = Path.cwd()
project_root = current_folder.parent.parent.parent
DATA_DIR = project_root / "data" / "raw"
ARTICLES_PATH = os.path.join(DATA_DIR, "articles.csv")
CUSTOMERS_PATH = os.path.join(DATA_DIR, "customers.csv")
TRANSACTIONS_PATH = os.path.join(DATA_DIR, "transactions_train.csv")

# Ensure IDs are strings for safe joins
articles_df = pd.read_csv(ARTICLES_PATH, dtype={"article_id": "string"})
customers = pd.read_csv(CUSTOMERS_PATH, dtype={"customer_id": "string"})
transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    dtype={"customer_id": "string", "article_id": "string"},
    parse_dates=["t_dat"],
)

print("articles:", articles_df.shape)
print("customers:", customers.shape)
print("transactions:", transactions.shape)


articles: (105542, 25)
customers: (1371980, 7)
transactions: (31788324, 5)


In [3]:
# Create Check Function
def check_image_exists(article_id):
    # Get the first 3 digit for the subfolder
    subfolder = article_id[:3]

    # Construct the image path
    image_path = os.path.join(DATA_DIR, "images", subfolder, f"{article_id}.jpg")

    return os.path.exists(image_path)

# Scan and Filter
print(f"Original articles count: {len(articles_df)}")
print("Scanning for images...")

# Check if images exist and filter the articles
articles_df["has_image"] = articles_df["article_id"].apply(check_image_exists)

# Filter articles that have images
clean_articles_df = articles_df[articles_df["has_image"]].copy()

# Drop the helper column
clean_articles_df.drop(columns=["has_image"], inplace=True)

print(f"Articles with images: {len(clean_articles_df)}")
print(f"Dropped articles without images: {len(articles_df) - len(clean_articles_df)}")

Original articles count: 105542
Scanning for images...
Articles with images: 105100
Dropped articles without images: 442


# Top-Bottom Filtering

In [4]:
# Ensure IDs are properly formatted strings
clean_articles_df["article_id"] = clean_articles_df["article_id"].apply(lambda x: f"{int(x):010d}")

# Determine Gender
is_ladies_index = clean_articles_df['index_name'].isin(['Ladieswear', 'Divided'])
is_mens_index = clean_articles_df['index_name'] == 'Menswear'

print("Articles with Ladieswear index:", is_ladies_index.sum())
print("Articles with Menswear index:", is_mens_index.sum())

gender_conditions = [is_ladies_index, is_mens_index]
gender_choices = ['Ladieswear', 'Menswear']
clean_articles_df['broad_gender'] = np.select(gender_conditions, gender_choices, default='other')

# Determine Type ('top' vs 'bottom')
type_conditions = [
    clean_articles_df['product_group_name'] == 'Garment Upper body',
    clean_articles_df['product_group_name'] == 'Garment Lower body'
]
type_choices = ['Top', 'Bottom']
clean_articles_df['broad_type'] = np.select(type_conditions, type_choices, default='other')

# 4. Combine them
clean_articles_df['combined_category'] = clean_articles_df['broad_gender'] + ' ' + clean_articles_df['broad_type']

# 5. Filter and Format
valid_items = clean_articles_df[
    (clean_articles_df['broad_gender'] != 'other') & 
    (clean_articles_df['broad_type'] != 'other')
].copy()

category_triplets = pd.DataFrame({
    'head': valid_items['article_id'],
    'relation' : 'in_category',
    'tail': valid_items['combined_category']
})

print(f"\nGenerated {len(category_triplets)} 'in_category' triplets.")
print("\nCategory Distribution:")
print(category_triplets['tail'].value_counts())

# Display the clean triplets
display(category_triplets.head(10))

Articles with Ladieswear index: 41035
Articles with Menswear index: 12504

Generated 39222 'in_category' triplets.

Category Distribution:
tail
Ladieswear Top       21056
Ladieswear Bottom     8863
Menswear Top          6844
Menswear Bottom       2459
Name: count, dtype: int64


,head,relation,tail
0,0108775015,in_category,Ladieswear Top
1,0108775044,in_category,Ladieswear Top
2,0108775051,in_category,Ladieswear Top
15,0116379047,in_category,Ladieswear Top
16,0118458003,in_category,Menswear Bottom
17,0118458004,in_category,Menswear Bottom
18,0118458028,in_category,Menswear Bottom
19,0118458029,in_category,Menswear Bottom
20,0118458034,in_category,Menswear Bottom
21,0118458038,in_category,Menswear Bottom


# Best Matches Based on Transaction

In [5]:
# Create a Category Lookup Dictionary
# Maps '0123456789' -> 'women top', 'mens bottom', etc.
category_dict = dict(zip(category_triplets['head'], category_triplets['tail']))

print("Mapping categories to transactions...")
# Apply the mapping. If an item isn't a top or bottom, it becomes NaN
transactions['category'] = transactions['article_id'].map(category_dict)

# Drop transactions for accessories, shoes, or unrecognized items
trans_filtered = transactions.dropna(subset=['category']).copy()

# Separate Tops and Bottoms
trans_tops = trans_filtered[trans_filtered['category'].str.contains('Top')].copy()
trans_bottoms = trans_filtered[trans_filtered['category'].str.contains('Bottom')].copy()

# Extract just the gender word (e.g., split 'women top' to get 'women')
trans_tops['gender'] = trans_tops['category'].str.split(' ').str[0]
trans_bottoms['gender'] = trans_bottoms['category'].str.split(' ').str[0]

print("Finding co-occurrences (Same customer, same date)...")
# Merge to find items bought together
merged = pd.merge(
    trans_tops[['customer_id', 't_dat', 'article_id', 'gender']],
    trans_bottoms[['customer_id', 't_dat', 'article_id', 'gender']],
    on=['customer_id', 't_dat'],
    suffixes=('_top', '_bottom')
)

# Enforce Gender Matching
# CRITICAL: Only keep pairs where both items are for the same gender
valid_pairs = merged[merged['gender_top'] == merged['gender_bottom']].copy()

print("Counting frequency of matching outfits...")
# Count the frequencies 
pair_counts = valid_pairs.groupby(
    ['article_id_top', 'article_id_bottom']
).size().reset_index(name='count')

# LIFT Calculation
print("Calculating Lift for pairs...")

# Get the total number of unique shopping baskets (customer_id + t_dat)
total_baskets = valid_pairs[['customer_id', 't_dat']].drop_duplicates().shape[0]

# Calculate how often each individual item is bought
top_frequencies = valid_pairs['article_id_top'].value_counts() / total_baskets
bottom_frequencies = valid_pairs['article_id_bottom'].value_counts() / total_baskets

# Map these overall frequencies back to the pairs
pair_counts['support_top'] = pair_counts['article_id_top'].map(top_frequencies)
pair_counts['support_bottom'] = pair_counts['article_id_bottom'].map(bottom_frequencies)

# Calculate the frequency of them being bought together (support of the pair)
pair_counts['support_pair'] = pair_counts['count'] / total_baskets

# Magic Formula: Lift = P(A and B) / (P(A) * P(B))
pair_counts['lift'] = pair_counts['support_pair'] / (pair_counts['support_top'] * pair_counts['support_bottom'])

# Sort by LIFT instead of Count
meaningful_pairs = pair_counts[pair_counts['lift'] > 1].copy()  # Only keep pairs that are bought together more than expected by chaance

# Sort to find the highest LIFT pairs
meaningful_pairs = meaningful_pairs.sort_values(['article_id_top', 'lift'], ascending=[True, False])

# Avoid the K-Core Trap
best_matches = meaningful_pairs.groupby('article_id_top').head(5)

# Format for the Knowledge Graph
match_edges = best_matches[['article_id_top', 'article_id_bottom', 'lift']].copy()
match_edges.columns = ['source', 'target', 'weight']  # Using Lift as the edge weight!
match_edges['relation'] = 'best_matches_with'

print(f"Generated {len(match_edges)} highly accurate 'best_matches_with' edges based on Lift.")
display(match_edges.head(10))

Mapping categories to transactions...
Finding co-occurrences (Same customer, same date)...
Counting frequency of matching outfits...
Calculating Lift for pairs...
Generated 125911 highly accurate 'best_matches_with' edges based on Lift.


,source,target,weight,relation
428,0108775015,0573552001,77.223002,best_matches_with
198,0108775015,0513402001,36.037401,best_matches_with
625,0108775015,0613967003,20.020778,best_matches_with
1339,0108775015,0708303002,16.892532,best_matches_with
52,0108775015,0396135045,11.261688,best_matches_with
1927,0108775044,0416511014,40.469989,best_matches_with
2250,0108775044,0588812001,36.42299,best_matches_with
1853,0108775044,0396135045,15.176246,best_matches_with
2018,0108775044,0532537001,15.176246,best_matches_with
2973,0108775044,0711473005,9.52235,best_matches_with


# Calculate Attribute Lift

In [6]:
# Grab the columns that we need form the articles dataframe
attributes = clean_articles_df[['article_id', 'colour_group_name', 'graphical_appearance_name', 'product_type_name']]

print("Attaching Top attributes...")
# Merge the attributes for TOP
valid_pairs_attr = pd.merge(
    valid_pairs,
    attributes,
    left_on='article_id_top',
    right_on='article_id',
    how='inner'
)

# Rename columns for clarity
valid_pairs_attr =valid_pairs_attr.rename(columns={
    'colour_group_name': 'color_top',
    'graphical_appearance_name': 'pattern_top',
    'product_type_name': 'type_top'
}).drop(columns=['article_id'])

print("Attaching Bottom attributes...")
# Merge the attributes for BOTTOM
valid_pairs_attr = pd.merge(
    valid_pairs_attr,
    attributes,
    left_on='article_id_bottom',
    right_on='article_id',
    how='inner'
)

# Rename columns for clarity
valid_pairs_attr = valid_pairs_attr.rename(columns={
    'colour_group_name': 'color_bottom',
    'graphical_appearance_name': 'pattern_bottom',
    'product_type_name': 'type_bottom'
}).drop(columns=['article_id'])

print("Clean columns:", valid_pairs_attr.columns.tolist())

Attaching Top attributes...
Attaching Bottom attributes...
Clean columns: ['customer_id', 't_dat', 'article_id_top', 'gender_top', 'article_id_bottom', 'gender_bottom', 'color_top', 'pattern_top', 'type_top', 'color_bottom', 'pattern_bottom', 'type_bottom']


In [21]:
# Lift Function
def calculate_attribute_lift(df, top_col, bottom_col, total_baskets_count):
    print(f"Calculating lift for columns {top_col} and {bottom_col}")

    # Calculate the frequencies of the attributes pairs
    pair_counts = df.groupby([top_col, bottom_col]).size().reset_index(name='count')

    # Calculate how often each individual attribute is bought
    top_frequencies = df[top_col].value_counts() / total_baskets_count
    bottom_frequencies = df[bottom_col].value_counts() / total_baskets_count

    # Map the overall frequencies back to the pairs
    pair_counts['support_top'] = pair_counts[top_col].map(top_frequencies)
    pair_counts['support_bottom'] = pair_counts[bottom_col].map(bottom_frequencies)

    # Calculate the frequency of them being bought together (support of the pair)
    pair_counts['support_pair'] = pair_counts['count'] / total_baskets_count

    # Magic Formula: Lift = P(A and B) / (P(A) * P(B))
    pair_counts['lift'] = pair_counts['support_pair'] / (pair_counts['support_top'] * pair_counts['support_bottom'])

    # Filter and sort
    #meaningful_pairs = pair_counts[pair_counts['lift'] > 1.0].copy()  

    # meaningful_pairs = meaningful_pairs.sort_values(['lift'], ascending=False)

    meaningful_pairs = pair_counts.sort_values(by=['lift'], ascending=[False])

    return meaningful_pairs[[top_col, bottom_col, 'count', 'lift']]

In [22]:
# Generate the matrices
total_baskets = valid_pairs_attr[['customer_id', 't_dat']].drop_duplicates().shape[0]

# Color Lift
color_lift = calculate_attribute_lift(valid_pairs_attr, 'color_top', 'color_bottom', total_baskets)
print("\nTop Color Lifts:")
print(color_lift.head(10))

# Pattern Lift
pattern_lift = calculate_attribute_lift(valid_pairs_attr, 'pattern_top', 'pattern_bottom', total_baskets)
print("\nTop Pattern Lifts:")
print(pattern_lift.head(10))

# Type Lift
type_lift = calculate_attribute_lift(valid_pairs_attr, 'type_top', 'type_bottom', total_baskets)
print("\nTop Type Lifts:")
print(type_lift.head(10))

Calculating lift for columns color_top and color_bottom

Top Color Lifts:
          color_top   color_bottom  count       lift
147   Bronze/Copper  Bronze/Copper     36  75.929167
1505     Other Blue     Other Blue    136  47.590412
1753   Other Yellow   Other Yellow   1266  17.765537
1940         Silver         Silver    917  17.215688
1638     Other Pink     Other Pink    178  16.068766
1593   Other Orange   Other Orange    254   9.100591
173   Bronze/Copper     Other Blue      4   8.717793
667            Gold         Orange      3   7.165083
1549    Other Green    Other Green      7   6.659696
1654   Other Purple  Bronze/Copper      1   6.647204
Calculating lift for columns pattern_top and pattern_bottom

Top Pattern Lifts:
             pattern_top       pattern_bottom  count      lift
561               Sequin               Sequin   2137  9.166019
405             Metallic             Metallic    188  4.503814
568                 Slub               Argyle      8  3.910200
687        

In [25]:
# Export to CSV
OUTPUT_DIR = project_root / "data" / "processed"
color_lift.to_csv(OUTPUT_DIR / "color_lift.csv", index=False)
pattern_lift.to_csv(OUTPUT_DIR / "pattern_lift.csv", index=False)
type_lift.to_csv(OUTPUT_DIR / "type_lift.csv", index=False)

# Price

In [9]:
# Group transaction to find the price of each item
item_price = transactions.groupby('article_id')['price'].mean().reset_index()

# Ensure the article_id is in 10-digit string format
item_price['article_id'] = item_price['article_id'].astype(str).str.zfill(10)

# Drop the price column from the original articles dataframe to avoid confusion
if 'price' in valid_items.columns:
    valid_items.drop(columns=['price'], inplace=True)

# Merge the prices into main article dataframe
valid_items = pd.merge(valid_items, item_price, on='article_id', how='left')

# Fill missing values (which is not in the transaction table)
global_median_price = valid_items['price'].median()
valid_items['price'] = valid_items['price'].fillna(global_median_price)

# This scales the tiny decimal back up to realistic H&M Singapore store prices
sgd_multiplier = 1200

# Apply to the price column
valid_items['price_sgd'] = (valid_items['price'] * sgd_multiplier).round(2)

print(valid_items[['article_id', 'price', 'price_sgd']].head(10))


   article_id     price  price_sgd
0  0108775015  0.008142       9.77
1  0108775044  0.008114       9.74
2  0108775051  0.004980       5.98
3  0116379047  0.011090      13.31
4  0118458003  0.022337      26.80
5  0118458004  0.027317      32.78
6  0118458028  0.024815      29.78
7  0118458029  0.030275      36.33
8  0118458034  0.014672      17.61
9  0118458038  0.032024      38.43


# Define Occasion & Seasons

In [10]:
# Extract Clean the Description Columns
print("Extracting Descriptions.....")
valid_items['clean_desc'] = valid_items['prod_name'].fillna('') + " - " + valid_items['detail_desc'].fillna('')

Extracting Descriptions.....


In [11]:
# Loading the model
print("Loading SentenceTransformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading SentenceTransformer model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 993.04it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
# Define the Target Categories ---
occasion_prompts = {
    'Sport': 'Activewear, gym, running, workout, sports, training, breathable, athletic, yoga',
    'Formal': 'Formal wear, business attire, office, suit, tailored, blazer, elegant dress, evening wear',
    'Casual': 'Casual everyday wear, relaxed, comfortable, denim, t-shirt, basic, lounge',
    'Party': 'Party wear, going out, clubbing, sequin, festive, glamour, cocktail dress, nightlife'
}
occasion_labels = list(occasion_prompts.keys())
occasion_vectors = model.encode(list(occasion_prompts.values()))

season_prompts = {
    'Spring': 'Spring, floral, light jacket, pastel, transitional weather, breezy, cotton, cardigan',
    'Summer': 'Summer, hot weather, breathable, linen, shorts, swimwear, tank top, sandals, beach',
    'Autumn': 'Autumn, fall, layers, knitwear, flannel, trench coat, light sweater, boots',
    'Winter': 'Winter, cold weather, heavy coat, puffer, snow, thermal, wool, warm, freezing'
}
season_labels = list(season_prompts.keys())
season_vectors = model.encode(list(season_prompts.values()))

In [13]:
# Generate Product Embeddings (Run Only Once)
descriptions_list = valid_items['clean_desc'].tolist()

print(f"Embedding {len(descriptions_list)} product descriptions...")
product_vectors = model.encode(descriptions_list, show_progress_bar=True)

Embedding 39222 product descriptions...


Batches: 100%|██████████| 1226/1226 [03:44<00:00,  5.45it/s]


In [14]:
# Calculate Cosine Similarity for both Occasion and Season
print('Calculating similarity scores...')
# Occasion Similarity
occ_similarity_matrix = cosine_similarity(product_vectors, occasion_vectors)
best_occ_indices = np.argmax(occ_similarity_matrix, axis=1)

# Season Similarity
season_similarity_matrix = cosine_similarity(product_vectors, season_vectors)
best_season_indices = np.argmax(season_similarity_matrix, axis=1)

# Assign to DataFrame
# Occasion Columns
valid_items['assigned_occasion'] = [occasion_labels[idx] for idx in best_occ_indices]
valid_items['occ_confidence_score'] = np.max(occ_similarity_matrix, axis=1)
# Season Columns
valid_items['assigned_season'] = [season_labels[idx] for idx in best_season_indices]
valid_items['season_confidence_score'] = np.max(season_similarity_matrix, axis=1)

print("\n--- Classification Complete! ---")
display(valid_items[['article_id', 'clean_desc', 'assigned_occasion', 'assigned_season', 'occ_confidence_score', 'season_confidence_score']].head(10))



Calculating similarity scores...

--- Classification Complete! ---


,article_id,clean_desc,assigned_occasion,assigned_season,occ_confidence_score,season_confidence_score
0,0108775015,Strap top - Jersey top with narrow shoulder st...,Casual,Summer,0.306843,0.319117
1,0108775044,Strap top - Jersey top with narrow shoulder st...,Casual,Summer,0.306843,0.319117
2,0108775051,Strap top (1) - Jersey top with narrow shoulde...,Casual,Summer,0.309744,0.342323
3,0116379047,Frugan longsleeve - Fitted top in soft stretch...,Casual,Summer,0.382151,0.425123
4,0118458003,Jerry jogger bottoms - Trousers in sweatshirt ...,Casual,Autumn,0.346937,0.357055
5,0118458004,Jerry jogger bottoms - Trousers in sweatshirt ...,Casual,Autumn,0.346937,0.357055
6,0118458028,Jerry jogger bottoms - Trousers in sweatshirt ...,Casual,Autumn,0.346937,0.357055
7,0118458029,Jerry jogger bottoms - Trousers in sweatshirt ...,Casual,Autumn,0.346937,0.357055
8,0118458034,Jerry jogger bottoms - Trousers in sweatshirt ...,Casual,Autumn,0.346937,0.357055
9,0118458038,Jerry jogger bottoms - Trousers in sweatshirt ...,Casual,Autumn,0.346937,0.357055


In [15]:
# Filter the DataFrame

# Category
df_category = pd.DataFrame({
    'head' : category_triplets['head'],
    'relation' : category_triplets['relation'],
    'tail' : category_triplets['tail'],
    'weight' : np.nan,
    'tail_color' : np.nan
})

# Color
df_color = pd.DataFrame({
    'head' : valid_items['article_id'],
    'relation' : 'has_color',
    'tail' : valid_items['colour_group_name'],
    'weight' : np.nan,
    'tail_color' : np.nan,
})

# Best Matches
df_matches = pd.DataFrame({
    'head' : match_edges['source'],
    'relation' : match_edges['relation'],
    'tail' : match_edges['target'],
    'weight' : match_edges['weight'],  # Using Lift as the weight
    'tail_color' : match_edges['target'].map(valid_items.set_index('article_id')['colour_group_name'])
})

# Price
df_price = pd.DataFrame({
    'head' : valid_items['article_id'],
    'relation' : 'has_price',
    'tail' : valid_items['price_sgd'],  # Using the scaled price in
    'weight' : np.nan,
    'tail_color' : np.nan
})

# Occasions
df_occasion = pd.DataFrame({
    'head' : valid_items['article_id'],
    'relation' : 'has_occasion',
    'tail' : valid_items['assigned_occasion'],
    'weight' : valid_items['occ_confidence_score'],
    'tail_color' : np.nan
})

# Seasons
df_season = pd.DataFrame({
    'head' : valid_items['article_id'],
    'relation' : 'has_season',
    'tail' : valid_items['assigned_season'],
    'weight' : valid_items['season_confidence_score'],
    'tail_color' : np.nan
})

# Concatenate all edges into one DataFrame
final_edges = pd.concat([df_category, df_color, df_matches, df_price, df_occasion, df_season], ignore_index=True)

# Clean up the DataFrame (e.g., drop rows with missing heads or tails)
final_edges = final_edges.dropna(subset=['head', 'tail']).reset_index(drop=True)
final_edges['head'] = final_edges['head'].astype(str).str.zfill(10)
final_edges = final_edges.sort_values(by=['head',], ascending=[True])
display(final_edges.head(10))


,head,relation,tail,weight,tail_color
0,0108775015,in_category,Ladieswear Top,<NA>,NaN
204355,0108775015,has_price,9.77,<NA>,NaN
243577,0108775015,has_occasion,Casual,0.306843,NaN
39222,0108775015,has_color,Black,<NA>,NaN
282799,0108775015,has_season,Summer,0.319117,NaN
78444,0108775015,best_matches_with,0573552001,77.223002,Black
78445,0108775015,best_matches_with,0513402001,36.037401,Dark Blue
78446,0108775015,best_matches_with,0613967003,20.020778,Dark Grey
78448,0108775015,best_matches_with,0396135045,11.261688,Yellow
78447,0108775015,best_matches_with,0708303002,16.892532,Black


# -----------------------------------------------------------------------

# -----------------------------------------------------------------------

In [16]:
# Export to CSV
OUTPUT_DIR = project_root / "data" / "processed"
# Export the dataframe to a CSV file in your current directory
final_edges.to_csv(OUTPUT_DIR / "final_knowledge_graph.csv", index=False)

print("✅ CSV successfully exported!")

✅ CSV successfully exported!


In [17]:
import joblib

joblib.dump(clean_articles_df, 'clean_articles_df.joblib')

['clean_articles_df.joblib']

In [18]:
import joblib

recommendation_model = joblib.load('clean_articles_df.joblib')